In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Conv1D, BatchNormalization, Activation, AveragePooling1D, LSTM, Add, Dense
from tensorflow.keras.models import Model

# 1. Config parameters
history_lengths = [42, 78, 150, 294, 582]
pooling_widths  = [3, 6, 12, 24, 48]
conv_filters    = [32, 32, 32, 32, 32]
conv_widths     = [7, 7, 7, 7, 7]
embedding_dims  = 32
lstm_units      = 32
hidden_neurons  = [128, 128]
conv_act        = 'relu'
hidden_act      = 'relu'
VOCAB_SIZE      = 4096

# 2. Build a slice
def build_slice(hist_len, pool_width, conv_filter, conv_width, tag):
    inp = Input(shape=(hist_len,), dtype='int32', name=f'{tag}_in')
    x = Embedding(VOCAB_SIZE, embedding_dims, name=f'{tag}_emb')(inp)
    x = Conv1D(conv_filter, conv_width, padding='valid', name=f'{tag}_conv')(x)
    x = BatchNormalization(name=f'{tag}_bn')(x)
    x = Activation(conv_act, name=f'{tag}_act')(x)
    x = AveragePooling1D(pool_width, name=f'{tag}_pool')(x)
    x = LSTM(lstm_units, name=f'{tag}_lstm')(x)
    x = BatchNormalization(name=f'{tag}_lstm_bn')(x)
    x = Activation('tanh', name=f'{tag}_tanh')(x)
    return inp, x

# 3. Build all slices
inputs, vectors = zip(*[
    build_slice(history_lengths[i], pooling_widths[i], conv_filters[i], conv_widths[i], f'slice{i}')
    for i in range(5)
])

# 4. Folded Add (for hls4ml compatibility)
merged = vectors[0]
for i, v in enumerate(vectors[1:], start=1):
    merged = Add(name=f'add_fold_{i}')([merged, v])

# 5. FC Head
x = merged
for i, n in enumerate(hidden_neurons):
    x = Dense(n, name=f'fc_{i}')(x)
    x = BatchNormalization(name=f'fc_{i}_bn')(x)
    x = Activation(hidden_act, name=f'fc_{i}_act')(x)
output = Dense(1, activation='sigmoid', name='output')(x)

model = Model(inputs=list(inputs), outputs=output, name='BranchNet_Keras_Modular')
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "BranchNet_Keras_Modular"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 slice0_in (InputLayer)      [(None, 42)]                 0         []                            
                                                                                                  
 slice1_in (InputLayer)      [(None, 78)]                 0         []                            
                                                                                                  
 slice0_emb (Embedding)      (None, 42, 32)               131072    ['slice0_in[0][0]']           
                                                                                                  
 slice1_emb (Embedding)      (None, 78, 32)               131072    ['slice1_in[0][0]']           
                                                                            

In [4]:
import tensorflow as tf
import yaml

# Load config file
with open('/home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/KladosNet/branchnet/configs/BranchNetLSTM.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# For SINGLE slice:
HIST_LEN_L0 = 44
VOCAB_SIZE = 128

# Dummy input (for warmup, summary, etc)
# dummy_input = tf.random.uniform((1, HIST_LEN_L0), minval=0, maxval=VOCAB_SIZE, dtype=tf.int32)
# model(dummy_input)

from dataloader_tf import BranchTraceDatasetTF

# Parameters for loader
trace_paths = ['../traces/641.leela_s-602B_dataset.hdf5']
br_pc = 4320802
history_length = history_lengths  # Now it's a single value, not a list
pc_bits = 11
pc_hash_bits = 7
hash_dir_with_pc = True

# Create Dataset
dataset_loader = BranchTraceDatasetTF(
    trace_paths=trace_paths,
    br_pc=br_pc,
    history_lengths=history_length,  # Provide a list here, even if it's a single value
    pc_bits=pc_bits,
    pc_hash_bits=pc_hash_bits,
    hash_dir_with_pc=hash_dir_with_pc
)
dataset = dataset_loader.get_dataset()#batch_size=128
model.fit(dataset, epochs=1)

     55/Unknown - 31s 266ms/step - loss: 0.0595 - accuracy: 0.9858

KeyboardInterrupt: 

In [5]:
# Suppose you have test_traces, test_br_pc, etc.
test_trace_paths = ['../traces/641.leela_s-862B_dataset.hdf5']

test_dataset_loader = BranchTraceDatasetTF(
    trace_paths=test_trace_paths,
    br_pc=4320802,  # use same branch PC or new one as needed
    history_lengths=history_lengths,
    pc_bits=pc_bits,
    pc_hash_bits=pc_hash_bits,
    hash_dir_with_pc=hash_dir_with_pc
)
test_dataset = test_dataset_loader.get_dataset()  # batch_size=128

results = model.evaluate(test_dataset)
print('Test loss, Test accuracy:', results)

    114/Unknown - 28s 211ms/step - loss: 0.5383 - accuracy: 0.9388

KeyboardInterrupt: 

In [6]:
import hls4ml

# 1) build the base config
cfg = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')

# 2) set your global defaults
cfg['Model']['Precision']   = 'ap_fixed<16,6>'
cfg['Model']['ReuseFactor'] = 1

# 3) loop over every layer in the config
for layer_name, layer_cfg in cfg.items():
    # skip the top‐level Model section
    if layer_name == 'Model':
        continue

    ln = layer_name.lower()
    #  ── any conv, lstm or dense layer gets 8,3 ──
    if 'conv' in ln or 'lstm' in ln or 'dense' in ln:
        # if Precision is a dict you may need:
        #   layer_cfg['Precision']['weight'] = 'ap_fixed<8,3>'
        #   layer_cfg['Precision']['bias']   = 'ap_fixed<8,3>'
        # else you can simply override the whole thing:
        layer_cfg['Precision']   = 'ap_fixed<8,3>'
        layer_cfg['ReuseFactor'] = 1

# 4) now you can regenerate the HLS model
hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=cfg, output_dir='BranchNet_HLS',
    part='xcu250-figd2104-2L-e', backend='Vitis')#xcu250-figd2104-2L-e

hls_model.compile()

In [15]:
import numpy as np


# Initialize variables to store true labels and predictions
y_true = []
y_pred = []

# Loop through the test dataset and predict the labels using the HLS4ML model
for x_batch, y_batch in test_dataset:
    # x_batch is the input sequence (tuple of slices), y_batch is the true label
    y_true.extend(y_batch.numpy())  # Add the true labels to y_true
    
    # Unpack the x_batch tuple and cast to float32
    x_batch = [slice.numpy().astype(np.float32) for slice in x_batch]  # Convert each slice to float32
    
    # Run predictions using the HLS4ML model
    predictions = hls_model.predict(x_batch)
    
    # Round the predictions to 0 or 1 and add to y_pred
    y_pred.extend(np.round(predictions).flatten())

# Calculate accuracy
accuracy = np.mean(np.array(y_true) == np.array(y_pred)) * 100

print(f'Accuracy: {accuracy:.2f}%')

KeyboardInterrupt: 

In [17]:
# Calculate accuracy
accuracy = np.mean(np.array(y_true[:50304]) == np.array(y_pred[:50304])) * 100

print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 25.32%


In [ ]:
# from sklearn.metrics import accuracy_score

# y_pred_hls = hls_model.predict(test_dataset)               # C-simulation

# # acc_hls    = accuracy_score(y_val.ravel(), y_pred_hls.ravel() > 0.5)
# # print("HLS model accuracy:", acc_hls)

In [7]:
hls_model.build(csim=True, synth=False)           # now calls vitis_hls

Exception: Vitis installation not found. Make sure "vitis-run" is on PATH.

In [6]:
hls4ml.report.read_vivado_report('/home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/BranchNet_HLS')

Found 1 solution(s) in /home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/BranchNet_HLS/myproject_prj.
Reports for solution "solution1":

C SIMULATION RESULT:
INFO: [SIM 2] *************** CSIM start ***************
INFO: [SIM 4] CSIM will launch GCC as the compiler.
   Compiling ../../../../myproject_test.cpp in debug mode
   Compiling ../../../../firmware/myproject.cpp in debug mode
   Generating csim.exe
/tools/Xilinx/Vivado/2022.2/tps/lnx64/binutils-2.37/bin/ld: /lib/x86_64-linux-gnu/libm.so.6: unknown type [0x13] section `.relr.dyn'
/tools/Xilinx/Vivado/2022.2/tps/lnx64/binutils-2.37/bin/ld: skipping incompatible /lib/x86_64-linux-gnu/libm.so.6 when searching for /lib/x86_64-linux-gnu/libm.so.6
/tools/Xilinx/Vivado/2022.2/tps/lnx64/binutils-2.37/bin/ld: cannot find /lib/x86_64-linux-gnu/libm.so.6
/tools/Xilinx/Vivado/2022.2/tps/lnx64/binutils-2.37/bin/ld: /lib/x86_64-linux-gnu/libm.so.6: unknown type [0x13] section `.relr.dyn'
/too

In [9]:
import hls4ml

config = hls4ml.backends.get_backend('VivadoAccelerator').create_initial_config()
print(config)

{'Part': 'xcvu13p-flga2577-2-e', 'ClockPeriod': 5, 'ClockUncertainty': '12.5%', 'IOType': 'io_parallel', 'HLSConfig': {}, 'WriterConfig': {'Namespace': None, 'WriteWeightsTxt': True, 'WriteTar': False}, 'AcceleratorConfig': {'Board': 'pynq-z2', 'Interface': 'axi_stream', 'Driver': 'python', 'Precision': {'Input': 'float', 'Output': 'float'}}}


In [10]:
# hls_model.build(csim=False, export=True, bitfile=True)
hls_model.build(csim=False, export=True)


****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2022.2 (64-bit)
  **** SW Build 3670227 on Oct 13 2022
  **** IP Build 3669848 on Fri Oct 14 08:30:02 MDT 2022
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis_HLS/2022.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/tools/Xilinx/Vitis_HLS/2022.2/bin/unwrapped/lnx64.o/vitis_hls'
INFO: [HLS 200-10] For user 'gkapakos' on host 'gkapakos-Type1ProductConfigId' (Linux_x86_64 version 6.8.0-62-generic) on Wed Jul 02 18:00:53 EEST 2025
INFO: [HLS 200-10] On os Ubuntu 24.04.2 LTS
INFO: [HLS 200-10] In directory '/home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/BranchNet_HLS'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
INFO: [HLS 200-10] Opening project '/home/gkapakos/Desktop/ECE/10th_Semester/Architecture_of_Parallel_Systems/Project/BranchPredictionAI/BranchNet_HLS/myproje

In [11]:
print(hls_model.config.backend)